In [ ]:
# 03_random_forest.ipynb — Random Forest Baseline Model

## Purpose
Train and evaluate a Random Forest classifier to predict **30-day readmission** using the
UCI Diabetes 130-US Hospitals dataset. This notebook serves as a **nonlinear model baseline**
to compare against logistic regression.

## Inputs
- `outputs/06_model_dataset_full.csv`  
  Generated from the SQL pipeline (`run_query.py`) and `sql/06_model_dataset_full.sql`.

## Target
- `readmit_binary`  
  - 1 = readmitted within 30 days (`<30`)  
  - 0 = not readmitted (`NO` or `>30`)

## Features
Administrative + utilization features including:
- Demographics: `age`, `gender`, `race`
- Utilization/severity counts: `time_in_hospital`, `num_lab_procedures`, `num_procedures`,
  `num_medications`, `number_outpatient`, `number_emergency`, `number_inpatient`, `number_diagnoses`
- Lab summaries: `A1Cresult`, `max_glu_serum`
- Treatment signals: `insulin`, `change`, `diabetesMed`
- Administrative codes: `admission_type_id`, `discharge_disposition_id`, `admission_source_id`

## Modeling Notes
- Random Forest is used to capture **nonlinear effects and interactions** automatically.
- Preprocessing:
  - One-hot encoding for categorical features
  - Median/most-frequent imputation for missing values
  - No scaling needed for tree models

## Evaluation
Primary metrics:
- ROC AUC (ranking/discrimination)
- PR AUC / Average Precision (useful with class imbalance)
Optional operational metric:
- “Top X% risk” enrichment (readmission rate within highest-risk segment)

## Output
Printed metrics and (optionally) threshold-based analysis for comparison to the logistic model.

In [1]:
import pandas as pd
import numpy as np

rf_df = pd.read_csv(
    r"C:\Users\palla\OneDrive\Documents\Coding Projects\Hospital Diabetes Dataset\hospital-readmissions-sql-eda\outputs\06_model_dataset_full.csv"
)

print(rf_df.shape)
print(rf_df.head())

(101766, 20)
   readmit_binary      age  gender             race  time_in_hospital  \
0               0   [0-10)  Female        Caucasian                 1   
1               0  [10-20)  Female        Caucasian                 3   
2               0  [20-30)  Female  AfricanAmerican                 2   
3               0  [30-40)    Male        Caucasian                 2   
4               0  [40-50)    Male        Caucasian                 1   

   num_lab_procedures  num_procedures  num_medications  number_outpatient  \
0                  41               0                1                  0   
1                  59               0               18                  0   
2                  11               5               13                  2   
3                  44               1               16                  0   
4                  51               0                8                  0   

   number_emergency  number_inpatient  number_diagnoses A1Cresult  \
0               

In [2]:
rf_df = rf_df.replace(r"^\s*$", np.nan, regex=True)

In [3]:
X = rf_df.drop(columns=["readmit_binary"])
y = rf_df["readmit_binary"].astype(int)

numeric_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

categorical_cols = [c for c in X.columns if c not in numeric_cols]

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", rf_model)
])

rf_pipeline.fit(X_train, y_train)

rf_proba = rf_pipeline.predict_proba(X_test)[:,1]

print("Random Forest ROC AUC:", round(roc_auc_score(y_test, rf_proba),4))
print("Random Forest PR AUC:", round(average_precision_score(y_test, rf_proba),4))

Random Forest ROC AUC: 0.6523
Random Forest PR AUC: 0.2094


In [ ]:
# Summary of Findings — Random Forest vs Logistic Regression

## Objective
Evaluate whether a nonlinear tree-based model (Random Forest) improves predictive performance
over logistic regression for 30-day readmission risk.

## Results

Logistic Regression (baseline):
- ROC AUC ≈ 0.675
- PR AUC ≈ 0.219

Random Forest:
- ROC AUC ≈ 0.652
- PR AUC ≈ 0.209

## Interpretation

The Random Forest model did **not** outperform logistic regression. This suggests that:

- The predictive structure in this administrative dataset is largely additive.
- Nonlinear thresholds and complex interactions do not substantially improve discrimination.
- A regularized linear model is sufficient to capture most of the available signal.

This is an important modeling insight: more complex models do not automatically yield better performance.
Model selection should be driven by empirical validation rather than algorithm complexity.

## Conclusion

Logistic regression remains the preferred model at this stage due to:

- Higher ROC AUC
- Comparable or better PR performance
- Greater interpretability
- Lower model complexity

Further improvements are more likely to come from feature engineering
or probability calibration rather than switching to more complex models.